# Session 1: 환경 세팅 & 데이터 탐색

## MLOps란?

**MLOps (Machine Learning Operations)** 는 머신러닝 모델을 개발부터 배포, 모니터링까지 체계적으로 관리하는 방법론입니다.

### 이 과정에서 만들 것

우리는 **대출 심사 자동화 시스템**을 처음부터 끝까지 구축합니다:

1. **데이터 탐색** (Session 1) - 데이터를 이해하고 인사이트 도출
2. **모델 학습** (Session 2) - XGBoost 모델 학습 및 Pipeline 구축
3. **API 기초** (Session 3) - FastAPI로 REST API 기본 구조 학습
4. **예측 API 구현** (Session 4) - 학습된 모델을 API로 서빙
5. **컨테이너화** (Session 5) - Docker로 패키징
6. **배포 & 모니터링** (Session 6) - 클라우드 배포 및 운영

### 왜 MLOps가 중요한가?

- 노트북에서 잘 동작하는 모델이 실제 서비스에서는 동작하지 않는 경우가 많습니다.
- 모델 재학습, 버전 관리, 성능 모니터링 등 운영 이슈를 체계적으로 다뤄야 합니다.
- 이 과정을 통해 **모델 개발 → 서빙 → 운영**의 전체 흐름을 경험합니다.

In [10]:
!pip install fastapi uvicorn scikit-learn xgboost pandas joblib matplotlib seaborn

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('환경 설정 완료!')

환경 설정 완료!


## 대출 심사 데이터 소개

### 비즈니스 시나리오

우리는 은행의 **대출 심사 자동화 시스템**을 구축하는 ML 엔지니어입니다.

현재 대출 심사는 심사역이 수동으로 처리하고 있어 시간이 오래 걸리고, 일관성이 부족합니다.  
ML 모델을 활용하여 대출 승인 여부를 예측하는 API를 만들어 심사 프로세스를 자동화하려 합니다.

### 데이터 컬럼 설명

| 컬럼 | 설명 | 타입 |
|------|------|------|
| 나이 | 신청자 나이 | 수치형 |
| 성별 | 남/여 | 범주형 |
| 연소득 | 연간 소득 (만원) | 수치형 |
| 근속연수 | 현 직장 근속 기간 (년) | 수치형 |
| 주거형태 | 자가/전세/월세 | 범주형 |
| 신용점수 | 신용 평가 점수 | 수치형 |
| 기존대출건수 | 기존 대출 수 | 수치형 |
| 연간카드사용액 | 연간 카드 사용 금액 (만원) | 수치형 |
| 부채비율 | 소득 대비 부채 비율 (%) | 수치형 |
| 대출신청액 | 신청한 대출 금액 (만원) | 수치형 |
| 대출목적 | 대출 사유 | 범주형 |
| 상환방식 | 원금균등/원리금균등 등 | 범주형 |
| 대출기간 | 대출 기간 (개월) | 수치형 |
| 승인여부 | 대출 승인=1, 거절=0 (타겟) | 이진 |

In [14]:
df = pd.read_csv("..\data\loan_data.csv")
df.sample(1).T

,821
나이,50
성별,남
연소득,3400
근속연수,8
주거형태,월세
신용점수,551
기존대출건수,2
연간카드사용액,2440
부채비율,33.6
대출신청액,3900


In [ ]:
# AI를 만들기 위해서는 학습을 시켜야 한다 
    

In [19]:
# 결측치 먼저 확인
df.isnull().sum().to_frame().T


,나이,성별,연소득,근속연수,주거형태,신용점수,기존대출건수,연간카드사용액,부채비율,대출신청액,대출목적,상환방식,대출기간,승인여부
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Target Data
round(df.승인여부.value_counts(normalize=True),2)*100

승인여부
1    61.0
0    39.0
Name: proportion, dtype: float64

In [25]:
# 여기에 코드를 작성하세요
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   나이       1500 non-null   int64  
 1   성별       1500 non-null   object 
 2   연소득      1500 non-null   int64  
 3   근속연수     1500 non-null   int64  
 4   주거형태     1500 non-null   object 
 5   신용점수     1500 non-null   int64  
 6   기존대출건수   1500 non-null   int64  
 7   연간카드사용액  1500 non-null   int64  
 8   부채비율     1500 non-null   float64
 9   대출신청액    1500 non-null   int64  
 10  대출목적     1500 non-null   object 
 11  상환방식     1500 non-null   object 
 12  대출기간     1500 non-null   int64  
 13  승인여부     1500 non-null   int64  
dtypes: float64(1), int64(9), object(4)
memory usage: 164.2+ KB


In [26]:
# 여기에 코드를 작성하세요
obj_cols = ['성별','주거형태','대출목적','상환방식']
df[obj_cols]

,성별,주거형태,대출목적,상환방식
0,여,전세,자동차,원금균등
1,남,월세,주택구입,원리금균등
2,남,월세,자동차,원리금균등
3,여,자가,전세자금,원리금균등
4,여,월세,생활비,원리금균등
...,...,...,...,...
1495,남,자가,생활비,원금균등
1496,남,자가,사업자금,원금균등
1497,남,전세,교육비,원리금균등
1498,남,자가,주택구입,원리금균등


In [33]:
# 여기에 코드를 작성하세요
df.성별.unique()
df.주거형태.unique()
df.대출목적.unique()
df.상환방식.unique()

array(['원금균등', '원리금균등', '만기일시'], dtype=object)

In [ ]:
# 여기에 코드를 작성하세요


array(['전세', '월세', '자가'], dtype=object)

## 데이터 탐색 인사이트 정리

### 발견한 주요 사항

1. **타겟 분포**: 승인여부의 분포를 확인하여 클래스 불균형 여부를 파악했습니다.
2. **신용점수**: 승인 그룹이 거절 그룹보다 신용점수가 높은 경향이 뚜렷합니다. → 중요한 피처
3. **연소득**: 승인 그룹의 연소득이 상대적으로 높습니다.
4. **부채비율**: 부채비율이 높을수록 거절될 가능성이 높습니다.
5. **범주형 변수**: 주거형태, 대출목적 등에 따라 승인율 차이가 존재합니다.

